# NumPy: rzadziej używane funkcje, które warto znać

**Problem:** pandas jest zbudowany na NumPy, ale wiele przydatnych funkcji NumPy zostaje w cieniu, bo pandas ma "swój" odpowiednik (`.clip()`, `.quantile()`, `.diff()`). Część z nich to czysta duplikacja — ale kilka z nich jest **szybszych, elastyczniejszych albo robi coś, czego pandas w ogóle nie ma wprost** (np. średnia ważona).

**Porównanie:** ta notatka nie powtarza tego, co już jest w innych notatkach o pandas (`groupby`, `pivot`, tekst) — skupia się na funkcjach NumPy jako **uzupełnieniu** warsztatu, z jawnym wskazaniem, gdzie mają przewagę nad odpowiednikiem pandas, a gdzie to tylko inny zapis tego samego.

**Kiedy stosować:** gdy operujesz na surowych tablicach (`np.ndarray`) bez potrzeby pełnego `DataFrame`; gdy potrzebna wektoryzowana logika warunkowa szybsza niż `.apply()`; gdy pandas nie ma wprost tego, co potrzebujesz (średnia ważona, partial sort).

## Setup

In [ ]:
import numpy as np
import pandas as pd

sales = np.array([1200, 800, 1500, 300, 2200, 950, 50, 1800])
units = np.array([12, 8, 15, 3, 22, 9, 1, 18])
region = np.array(["North", "North", "South", "South", "East", "East", "West", "West"])

sales, units, region

## Sekcja 1 — `np.where`: warunkowe przypisanie

Z **trzema** argumentami: `(warunek, jeśli_prawda, jeśli_fałsz)` — dokładnie jak `IF()` w Excelu, wektoryzowane. Z **jednym** argumentem (samym warunkiem) robi coś zupełnie innego — patrz Pułapka 1.

In [ ]:
np.where(sales > 1000, "wysoka", "niska")

## Sekcja 2 — `np.select`: wiele warunków naraz

Zagnieżdżanie `np.where` w `np.where` przy więcej niż dwóch kategoriach szybko robi się nieczytelne. `np.select(warunki, wybory, default=)` przyjmuje LISTĘ warunków i LISTĘ wyborów tej samej długości — pierwszy pasujący warunek wygrywa, `default` obsługuje przypadek, gdy żaden nie pasuje (patrz też Pułapka 3 — pomijanie `default` bywa niebezpieczne).

In [ ]:
conditions = [sales < 500, sales < 1500, sales >= 1500]
choices = ["niska", "średnia", "wysoka"]

np.select(conditions, choices, default="brak")

## Sekcja 3 — `np.logical_and`/`or`/`not`/`xor`

Dla dwóch warunków to dokładnie to samo co operatory `&`/`|`/`~` — kwestia czytelności. Prawdziwa przewaga ujawnia się przy **więcej niż dwóch** warunkach naraz: `np.logical_and.reduce([...])` przyjmuje listę dowolnej długości, bez zagnieżdżania nawiasów.

In [ ]:
# Dla dwóch warunków - identyczny efekt co operator &
print(np.logical_and(sales > 500, units > 10))
print((sales > 500) & (units > 10))

# Dla wielu warunków naraz - .reduce() zamiast łańcucha nawiasów
many_conditions = [sales > 100, units > 5, sales < 2000]
print(np.logical_and.reduce(many_conditions))

## Sekcja 4 — `np.maximum`/`np.minimum`: element po elemencie (NIE mylić z `np.max`/`np.min`)

To dwie zupełnie różne funkcje o bardzo podobnych nazwach:
- `np.maximum(a, b)` — porównuje DWIE tablice element-po-elemencie, zwraca tablicę tej samej długości.
- `np.max(a)` — REDUKUJE jedną tablicę do pojedynczej liczby.

`np.maximum`/`minimum` to naturalny sposób na "przynajmniej tyle" / "co najwyżej tyle" względem innej kolumny (nie stałej — do tego służy `np.clip`, Sekcja 5).

In [ ]:
target = np.array([1000, 1000, 1000, 1000, 2000, 1000, 1000, 2000])

print("np.maximum(sales, target) - element po elemencie:")
print(np.maximum(sales, target))

print("\nnp.max(sales) - REDUKCJA do jednej liczby:")
print(np.max(sales))

## Sekcja 5 — `np.clip`: ograniczenie do zakresu

W odróżnieniu od `np.maximum`/`minimum` (porównanie z inną tablicą), `np.clip(a, min, max)` ogranicza wartości do STAŁEGO zakresu — dokładny odpowiednik `Series.clip()` w pandas, tu na surowej tablicy.

In [ ]:
np.clip(sales, 500, 1500)

## Sekcja 6 — `np.quantile`/`np.percentile`: więcej kontroli niż pandas

Różnica między nimi to tylko skala (`quantile` 0–1, `percentile` 0–100). Przewaga nad `Series.quantile()`: parametr `method=` (dawniej `interpolation=`) z kilkoma strategiami — istotne przy małych/dyskretnych zbiorach, gdzie "dokładny" kwantyl nie trafia w żadną rzeczywistą obserwację.

In [ ]:
print(f"quantile (skala 0-1): {np.quantile(sales, 0.25)}")
print(f"percentile (skala 0-100): {np.percentile(sales, 25)}")
print()
for method in ["linear", "lower", "higher", "nearest", "midpoint"]:
    print(f"method='{method}': {np.quantile(sales, 0.25, method=method)}")

## Sekcja 7 — `np.digitize`: binowanie (alternatywa dla `pd.cut`)

Zwraca INDEKS przedziału, do którego trafia każda wartość — szybsze niż `pd.cut`, gdy potrzebujesz tylko numeru binu (bez pełnego obiektu `Categorical`), np. jako wejście do dalszej logiki wektoryzowanej.

In [ ]:
bins = [0, 500, 1000, 1500, 2000, 3000]
bin_idx = np.digitize(sales, bins)
labels = ["0-500", "500-1000", "1000-1500", "1500-2000", "2000-3000"]

print("Wartości:      ", sales)
print("Indeksy binów: ", bin_idx)
print("Etykiety:      ", [labels[i - 1] for i in bin_idx])

## Sekcja 8 — `np.searchsorted`: szybkie wyszukiwanie w posortowanej tablicy

Zwraca pozycję, na której dana wartość zostałaby wstawiona, żeby zachować sortowanie. **Trik:** to naturalny sposób na "VLOOKUP z przybliżeniem w dół" — np. dopasowanie ceny do progu ilościowego, bez pisania pętli ani `.apply()`.

In [ ]:
# Trik: cena za sztukę zależna od progu zamówionej ilości (im więcej, tym taniej)
qty_breaks = np.array([0, 10, 50, 100])
prices = np.array([100, 90, 80, 70])
order_qty = np.array([5, 25, 75, 150])

price_idx = np.searchsorted(qty_breaks, order_qty, side="right") - 1
matched_prices = prices[price_idx]

print(f"Zamówienia:     {order_qty}")
print(f"Dopasowane ceny: {matched_prices}")

## Sekcja 9 — `np.average`: średnia WAŻONA (czego pandas nie ma wprost)

`Series.mean()` nie przyjmuje wag. `np.average(a, weights=...)` tak — przydatne np. do średniej ceny ważonej ilością sprzedanych sztuk, zamiast zwykłej średniej z cen.

In [ ]:
weights = np.array([1, 1, 2, 1, 3, 1, 1, 2])

print(f"Zwykła średnia:  {np.mean(sales):.1f}")
print(f"Średnia ważona:  {np.average(sales, weights=weights):.1f}")

## Sekcja 10 — `np.diff`: różnice, z kontrolą rzędu

Odpowiednik `Series.diff()`, ale z parametrem `n=` do różnic WYŻSZEGO rzędu — `n=2` liczy różnicę różnic, czyli przyspieszenie zmiany (druga pochodna dyskretna), nie tylko samą zmianę.

In [ ]:
print(f"Wartości:              {sales}")
print(f"Różnica 1. rzędu:      {np.diff(sales)}")
print(f"Różnica 2. rzędu (n=2): {np.diff(sales, n=2)}")

## Sekcja 11 — `np.unique` z `return_counts`

Podobne do `value_counts()`, ale działa na surowej tablicy (nie wymaga `Series`) i zwraca gotową PARĘ tablic (unikalne wartości + liczności) zamiast obiektu z własnym indeksem — czasem wygodniejsze do dalszych operacji wektoryzowanych.

In [ ]:
values, counts = np.unique(region, return_counts=True)
dict(zip(values, counts))

## Sekcja 12 — `np.argpartition`: szybkie top-N BEZ pełnego sortowania

Gdy potrzebujesz tylko "N największych wartości" (nie ich dokładnej kolejności), pełne sortowanie (`np.argsort`) robi więcej pracy niż trzeba — sortuje WSZYSTKO, żeby dobrać się do kilku ostatnich. `np.argpartition` gwarantuje tylko, że N największych znajdzie się na właściwym końcu tablicy, bez sortowania ich między sobą — i jest za to szybsze.

In [ ]:
import time

rng = np.random.default_rng(2)
big = rng.integers(0, 1_000_000, 1_000_000)
k = 5

start = time.perf_counter()
top_k_full_sort = np.argsort(big)[-k:]
t_sort = time.perf_counter() - start

start = time.perf_counter()
top_k_partition = np.argpartition(big, -k)[-k:]
t_partition = time.perf_counter() - start

print(f"Pełne argsort:           {t_sort*1000:.2f} ms")
print(f"argpartition (top-{k}):    {t_partition*1000:.2f} ms")
print(f"Różnica: {t_sort/t_partition:.1f}x szybciej")
print(f"Te same wartości top-{k}: {sorted(big[top_k_full_sort]) == sorted(big[top_k_partition])}")

## Sekcja 13 — Benchmark: `np.select` vs `.apply()` z funkcją warunkową

Ten sam motyw co w innych notatkach: wektoryzowana alternatywa dla logiki `if/elif` w `.apply()`.

In [ ]:
n = 300_000
big_sales = rng.integers(0, 3000, n)
df_big = pd.DataFrame({"sales": big_sales})


def categorize(x):
    if x < 500:
        return "niska"
    elif x < 1500:
        return "średnia"
    return "wysoka"


start = time.perf_counter()
result_apply = df_big["sales"].apply(categorize)
t_apply = time.perf_counter() - start

start = time.perf_counter()
arr = df_big["sales"].to_numpy()
result_select = np.select([arr < 500, arr < 1500], ["niska", "średnia"], default="wysoka")
t_select = time.perf_counter() - start

print(f".apply(funkcja) z if/elif: {t_apply:.4f}s")
print(f"np.select (wektoryzowane): {t_select:.4f}s")
print(f"Różnica: {t_apply/t_select:.1f}x szybciej")
print(f"Wyniki identyczne: {(result_apply.to_numpy() == result_select).all()}")

## Sekcja 14 — Pułapki

### Pułapka 1 — `np.where` z JEDNYM argumentem zwraca krotkę INDEKSÓW, nie wartości

To nie jest skrócona forma trzyargumentowego `np.where` — to zupełnie inna funkcja o tej samej nazwie. Bez `jeśli_prawda`/`jeśli_fałsz`, `np.where(warunek)` zachowuje się jak `np.nonzero()` — zwraca pozycje, na których warunek jest `True`, zapakowane w krotkę (po jednej tablicy na wymiar).

In [ ]:
result_3args = np.where(sales > 1000, "wysoka", "niska")
result_1arg = np.where(sales > 1000)

print(f"np.where(warunek, x, y) -> tablica wartości: {result_3args}")
print(f"np.where(warunek)       -> {type(result_1arg).__name__} indeksów: {result_1arg}")

### Pułapka 2 — `np.maximum` vs `np.max`: nazwa różni się jedną literą, znaczenie jest zupełnie inne

Podsumowanie z Sekcji 4: `maximum`/`minimum` (l. mnoga formy, dwie tablice) porównuje element-po-elemencie; `max`/`min` (l. pojedyncza) redukuje jedną tablicę do liczby. Użycie jednego zamiast drugiego albo rzuci błąd (zła liczba argumentów), albo — gorzej — zwróci coś, co wygląda sensownie, ale znaczy coś innego.

### Pułapka 3 — `np.select` bez `default`: brakująca wartość cicho staje się `0`

Gdy żaden warunek nie pasuje (np. wartość źródłowa to `NaN`), a `choicelist` jest liczbowa, domyślny `default` to `0` — nie `NaN`, nie błąd, nie ostrzeżenie. Wygląda jak realna, policzona wartość zerowa, a w rzeczywistości oznacza "żaden warunek nie zadziałał".

In [ ]:
sales_with_nan = np.array([1200, 800, np.nan, 300])
conditions = [sales_with_nan < 500, sales_with_nan >= 500]
choices_numeric = [sales_with_nan * 0.1, sales_with_nan * 0.2]  # np. dwie różne stawki prowizji

print("Bez jawnego default - NaN cicho zamienia się w 0.0:")
print(np.select(conditions, choices_numeric))

print("\nZ jawnym default=np.nan - poprawnie widać brak dopasowania:")
print(np.select(conditions, choices_numeric, default=np.nan))

## Podsumowanie

| Zadanie | Funkcja NumPy | Odpowiednik / uwaga pandas |
|---|---|---|
| Warunkowe przypisanie (if/else wektorowo) | `np.where(warunek, x, y)` | `Series.where()`/`.mask()` robią podobnie, ale odwrotnie sparametryzowane |
| Wiele warunków naraz | `np.select(warunki, wybory, default=)` | brak bezpośredniego odpowiednika — czytelniejsze niż zagnieżdżone `np.where` |
| Wiele warunków logicznych naraz | `np.logical_and.reduce([...])` | brak odpowiednika 1:1 |
| Większa/mniejsza z DWÓCH tablic, element po elemencie | `np.maximum`/`np.minimum` | brak wprost — zwykle robi się przez `.where()`/`np.where` |
| Ograniczenie do zakresu | `np.clip(a, min, max)` | `Series.clip()` — to samo |
| Kwantyl z kontrolą metody interpolacji | `np.quantile(a, q, method=)` | `Series.quantile()` ma mniej opcji `method` |
| Binowanie do indeksów przedziałów | `np.digitize(a, bins)` | `pd.cut()` daje więcej (etykiety, `Categorical`), `digitize` jest szybszy |
| Przybliżone wyszukiwanie w posortowanej tablicy | `np.searchsorted(sorted_a, values)` | brak wprost — przydatne do "lookup po progu" |
| Średnia ważona | `np.average(a, weights=)` | **brak w pandas wprost** |
| Różnica wyższego rzędu | `np.diff(a, n=)` | `Series.diff()` nie ma parametru `n` |
| Unikalne wartości + liczności | `np.unique(a, return_counts=True)` | `Series.value_counts()` — podobne, inny kształt wyniku |
| Top-N bez pełnego sortowania | `np.argpartition(a, -k)[-k:]` | `Series.nlargest()` — wygodniejsze, ale wolniejsze na dużych danych |

**Wniosek:** żadna z tych funkcji nie zastępuje pandas — to dopełnienie warsztatu tam, gdzie surowa tablica NumPy wystarczy (szybciej) albo gdzie pandas czegoś nie ma wprost (średnia ważona, wyższe różnice). Jak wszędzie w tym repo: `np.where`/`np.select` bez pełnej listy argumentów nie rzucają błędu — dają wynik, który wygląda poprawnie, dopóki nie porówna się go z oczekiwaniem.